# Kaggle 2×T4 — all remaining reviewer stress tests

This notebook strengthens the **same workshop paper** without adding a new task or method. It runs:

1. exact FP32 **crossed-receiver** downstream-suffix JVP on the frozen B1 design;
2. `p`-only, `J`-only, and fixed-logit-shift controls to separate softmax/output geometry from downstream transformation;
3. crossed doses at 0.10, 0.25, 0.50, and 1.00;
4. held-out 72/24 receiver-fragility prediction;
5. anchor-row residual audit;
6. a descriptive PCA + 5-NN naturality proxy.

## Run plan

Use a **fresh Kaggle session with GPU T4 ×2 and Internet ON**. Run the notebook once for Qwen, download the result ZIP, then start a fresh T4×2 session and change the config to Mistral.

- Full audit: `N_DELTAS = 16`.
- Smoke test only: `N_DELTAS = 2`. Never put smoke-test numbers in the paper.
- The primary receiver-ranking summaries exclude the original-host anchor, leaving 15 alternative receivers/displacement.
- The exact audit is **FP32 downstream-suffix JVP from FP16-cached natural states**, not full-model FP32.
- If `DELTA_NORM_PROTOCOL_CHECK.csv` fails the 2% check, stop and send that file instead of using the results.

At the end download `R2_STRESS_<model>_D16_R16_FINAL_RESULTS.zip`.


## v2 loader fix

The original stress-test notebook could stop with `HTTP Error 404` before any GPU work because it
read frozen CSVs from hard-coded unauthenticated `raw.githubusercontent.com/<old-commit>/...` URLs.

This version does **not** change the experiment. It changes only file retrieval:

- checks attached Kaggle Dataset files first;
- then GitHub API at `main`;
- then the frozen provenance commit;
- optionally reads a Kaggle Secret called `GITHUB_TOKEN` or `GH_TOKEN` for a private repository;
- fails **before model loading** with a precise access message if none of those work.

If your repo is public, no token should be required.


In [ ]:
# EDIT ONLY THESE TWO LINES
MODEL_TO_RUN = "qwen3_8b"   # second fresh session: "mistral_7b"
N_DELTAS = 16               # smoke test only: 2


In [ ]:
# Kaggle 2xT4 reviewer stress tests for Transformer MLP Transplants Depend on the Receiving State
# Run once with MODEL_TO_RUN='qwen3_8b', then a fresh T4x2 session with 'mistral_7b'.

import sys, subprocess, importlib
REQ={"transformers":"transformers>=4.53,<5","accelerate":"accelerate>=1.2,<2","datasets":"datasets>=3,<4","scipy":"scipy","pandas":"pandas","sklearn":"scikit-learn","matplotlib":"matplotlib","tqdm":"tqdm"}
miss=[]
for m,s in REQ.items():
    try: importlib.import_module(m)
    except Exception: miss.append(s)
if miss: subprocess.check_call([sys.executable,"-m","pip","install","-q",*miss])

import os,gc,json,random,hashlib,zipfile,warnings,requests,io
from pathlib import Path
import numpy as np,pandas as pd,torch,matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr,rankdata
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM
from accelerate import dispatch_model
warnings.filterwarnings("ignore")

assert torch.cuda.is_available() and torch.cuda.device_count()>=2, "Select Kaggle GPU T4 x2"
for i in range(2):
    print(i,torch.cuda.get_device_name(i),f"{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")
    assert "T4" in torch.cuda.get_device_name(i).upper()
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
try: torch.set_float32_matmul_precision("highest")
except Exception: pass
os.environ["TOKENIZERS_PARALLELISM"]="false"; os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

# ===================== EDIT ONLY THIS FOR SECOND SESSION =====================
MODEL_TO_RUN=globals().get("MODEL_TO_RUN","qwen3_8b")
N_DELTAS=int(globals().get("N_DELTAS",16))
# ============================================================================
R=16; DOSES=(0.10,0.25,0.50,1.00); SEED=260826; SEQ_LEN=64; N_CONTEXTS=256; RAW_POOL_SIZE=320
REPO_OWNER="mnehan67"
REPO_NAME="neurips-2026"
# Try current public main first. If the repo is private, add a Kaggle Secret named GITHUB_TOKEN.
# The old commit is kept only as a fallback provenance ref.
GITHUB_REF="main"
REPO_COMMIT="c6f672f477d90542c34fdd51276157cd42641ba1"
RESULT_ROOT="reproducibility/results"
SPECS={
 "qwen3_8b":{"model_id":"Qwen/Qwen3-8B","layer":14,
   "b1":f"{RESULT_ROOT}/physical_crossing_v3/primary/qwen_b1_layer14_cross_receiver_physical.csv",
   "control":f"{RESULT_ROOT}/receiver_transformation_v1/qwen3_8b/control_panel_rows.csv",
   "split":f"{RESULT_ROOT}/receiver_transformation_v1/qwen3_8b/train_test_split.csv"},
 "mistral_7b":{"model_id":"mistralai/Mistral-7B-v0.3","layer":12,
   "b1":f"{RESULT_ROOT}/physical_crossing_v3/primary/mistral_b1_layer12_cross_receiver_physical.csv",
   "control":f"{RESULT_ROOT}/receiver_transformation_v1/mistral_7b/control_panel_rows.csv",
   "split":f"{RESULT_ROOT}/receiver_transformation_v1/mistral_7b/train_test_split.csv"}}
spec=SPECS[MODEL_TO_RUN]
OUT=Path("/kaggle/working")/f"R2_STRESS_{MODEL_TO_RUN}_D{N_DELTAS}_R{R}"; (OUT/"per_delta").mkdir(parents=True,exist_ok=True)
print("Output:",OUT)

def clear_cuda(): gc.collect(); torch.cuda.empty_cache()
def rho(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); ok=np.isfinite(x)&np.isfinite(y)
    return float(spearmanr(x[ok],y[ok]).statistic) if ok.sum()>=4 and np.std(x[ok])>0 and np.std(y[ok])>0 else np.nan
def fisher(v,p):
    pv=(p*v).sum(); return float(0.5*((p*v.square()).sum()-pv.square()))
def KL(z0,z1):
    lp=torch.log_softmax(z0.float(),-1); p=lp.exp(); lq=torch.log_softmax(z1.float(),-1); return float((p*(lp-lq)).sum())
def bootmed(x,B=4000,seed=SEED+91):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if not len(x): return (np.nan,np.nan,np.nan)
    rg=np.random.default_rng(seed); b=np.array([np.median(rg.choice(x,len(x),replace=True)) for _ in range(B)])
    return float(np.median(x)),float(np.quantile(b,.025)),float(np.quantile(b,.975))
def zip_checkpoint():
    zp=Path("/kaggle/working")/(OUT.name+"_CHECKPOINT.zip")
    if zp.exists(): zp.unlink()
    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file(): z.write(p,p.relative_to(OUT))
    print("checkpoint:",zp)


# ---------- Robust GitHub/local input loader ----------
# The previous notebook used hard-coded raw.githubusercontent.com URLs. Private repos
# and ref changes can make those return a misleading 404. This helper:
#   1) uses a matching file already attached as a Kaggle Dataset, if present;
#   2) tries GitHub API at current main;
#   3) tries the frozen provenance commit;
#   4) uses GITHUB_TOKEN / GH_TOKEN from Kaggle Secrets when needed.
#
# No scientific value is changed by this loader; it only makes the frozen CSV retrieval robust.

def _github_token():
    for key in ["GITHUB_TOKEN","GH_TOKEN"]:
        if os.environ.get(key):
            return os.environ[key]
    try:
        from kaggle_secrets import UserSecretsClient
        us=UserSecretsClient()
        for key in ["GITHUB_TOKEN","GH_TOKEN"]:
            try:
                v=us.get_secret(key)
                if v: return v
            except Exception:
                pass
    except Exception:
        pass
    return None

def _local_candidate(relpath):
    # Prefer exact relative structure under /kaggle/input if the user attached the repo/results as a Dataset.
    rel = Path(relpath)
    candidates=[]
    root=Path("/kaggle/input")
    if root.exists():
        for p in root.rglob(rel.name):
            s=str(p).replace("\\\\","/")
            # disambiguate same basenames (e.g. train_test_split.csv) by matching trailing parent names
            tail="/".join(rel.parts[-3:])
            if s.endswith(tail) or rel.name not in ["train_test_split.csv","control_panel_rows.csv"]:
                candidates.append(p)
    return candidates[0] if candidates else None

def load_repo_csv(relpath):
    cache_dir=OUT/"frozen_input_cache"
    cache_dir.mkdir(exist_ok=True)
    cache_path=cache_dir/Path(relpath).name

    if cache_path.exists():
        print("cached:",relpath)
        return pd.read_csv(cache_path)

    lp=_local_candidate(relpath)
    if lp is not None:
        print("local Kaggle input:",lp)
        df=pd.read_csv(lp)
        df.to_csv(cache_path,index=False)
        return df

    token=_github_token()
    headers={"Accept":"application/vnd.github.raw+json","User-Agent":"receiver-stress-audit"}
    if token:
        headers["Authorization"]=f"Bearer {token}"

    refs=[]
    for ref in [GITHUB_REF, REPO_COMMIT]:
        if ref and ref not in refs: refs.append(ref)

    errors=[]
    for ref in refs:
        api=f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{relpath}"
        try:
            rr=requests.get(api,headers=headers,params={"ref":ref},timeout=60)
            if rr.status_code==200:
                cache_path.write_bytes(rr.content)
                print(f"GitHub API OK: {relpath} @ {ref}")
                return pd.read_csv(cache_path)
            errors.append(f"{ref}: HTTP {rr.status_code} {rr.text[:180]}")
        except Exception as e:
            errors.append(f"{ref}: {type(e).__name__}: {e}")

    # Public raw fallback (useful if API has a transient issue)
    for ref in refs:
        raw=f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{ref}/{relpath}"
        try:
            rr=requests.get(raw,timeout=60)
            if rr.status_code==200:
                cache_path.write_bytes(rr.content)
                print(f"raw GitHub OK: {relpath} @ {ref}")
                return pd.read_csv(cache_path)
            errors.append(f"raw {ref}: HTTP {rr.status_code}")
        except Exception as e:
            errors.append(f"raw {ref}: {type(e).__name__}: {e}")

    raise RuntimeError(
        "Could not retrieve frozen input:\\n"
        f"  {relpath}\\n\\n"
        "This is a repository-access problem, not a GPU/model problem.\\n"
        "Fix one of these and rerun:\\n"
        "  A) make mnehan67/neurips-2026 publicly readable; OR\\n"
        "  B) add a Kaggle Secret named GITHUB_TOKEN with read access to the repo; OR\\n"
        "  C) attach the repo's reproducibility/results directory as a Kaggle Dataset.\\n\\n"
        "Attempts:\\n  " + "\\n  ".join(errors)
    )

# Preflight ONLY the small frozen tables needed by CPU checks.
# This intentionally happens before any model download/GPU work.
for _mk,_sp in SPECS.items():
    for _k in ["control","split","b1"]:
        _df=load_repo_csv(_sp[_k])
        print("preflight",_mk,_k,_df.shape)
        del _df
print("Frozen-input preflight PASS.")



In [ ]:
# ---------- CPU held-out generic-fragility audit ----------
def add_rank(d,col):
    d=d.copy(); d["r_"+col]=d.groupby("displacement_id")[col].transform(lambda x: pd.Series(rankdata(x.to_numpy(float)),index=x.index)); return d
frag=[]
for mk,sp in SPECS.items():
    d=load_repo_csv(sp["control"]).merge(load_repo_csv(sp["split"]),on="displacement_id",validate="many_to_one")
    d=add_rank(add_rank(d,"kl_full"),"Fisher"); tr=d[d["split"]=="train"].copy(); te=d[d["split"]=="test"].copy()
    bk=tr.groupby("receiver_id")["r_kl_full"].mean(); bq=tr.groupby("receiver_id")["r_Fisher"].mean(); te["bk"]=te.receiver_id.map(bk); te["bq"]=te.receiver_id.map(bq)
    rr=[]
    for did,g in te.groupby("displacement_id"): rr.append([did,rho(g.bk,g.kl_full),rho(g.Fisher,g.kl_full)])
    rr=pd.DataFrame(rr,columns=["delta_id","fragility_rho","Fisher_rho"]); te["Kres"]=te.r_kl_full-te.bk; te["Qres"]=te.r_Fisher-te.bq
    m,lo,hi=bootmed(rr.Fisher_rho-rr.fragility_rho,seed=SEED+331)
    frag.append({"model":mk,"train_d":tr.displacement_id.nunique(),"test_d":te.displacement_id.nunique(),"median_fragility_rho":rr.fragility_rho.median(),"median_Fisher_rho":rr.Fisher_rho.median(),"median_Fisher_minus_fragility":m,"diff_CI_lo":lo,"diff_CI_hi":hi,"heldout_residual_rho":rho(te.Qres,te.Kres)})
pd.DataFrame(frag).to_csv(OUT/"HELDOUT_FRAGILITY_SUMMARY.csv",index=False); print("\nHeld-out fragility\n",pd.DataFrame(frag))



In [ ]:
# ---------- CPU anchor-row residual audit ----------
def twoway(v,a,b,it=200,tol=1e-10):
    r=np.asarray(v,float)-np.mean(v); a=np.asarray(a); b=np.asarray(b)
    for _ in range(it):
        old=r.copy(); t=pd.DataFrame({"r":r,"a":a,"b":b}); r-=t.groupby("a").r.transform("mean").to_numpy(); t["r"]=r; r-=t.groupby("b").r.transform("mean").to_numpy()
        if np.max(np.abs(r-old))<tol: break
    return r
def residual_rho(d,drop):
    if drop: d=d[d.receiver!=d.origin_host].copy()
    return rho(twoway(rankdata(d.q_fisher),d.delta_id,d.receiver),twoway(rankdata(d.kl_cross),d.delta_id,d.receiver)),len(d)
ar=[]
for mk,sp in SPECS.items():
    d=load_repo_csv(sp["b1"]); a,n1=residual_rho(d,False); b,n2=residual_rho(d,True); ar.append({"model":mk,"with_anchor":a,"without_anchor":b,"difference":b-a,"n_with":n1,"n_without":n2})
pd.DataFrame(ar).to_csv(OUT/"ANCHOR_ROW_RESIDUAL_AUDIT.csv",index=False); print("\nAnchor audit\n",pd.DataFrame(ar))



In [ ]:
# ---------- Reconstruct exact frozen B1 pool ----------
def get_b1():
    p=OUT/"c4_b1.jsonl"
    if p.exists():
        q=[json.loads(x) for x in p.read_text().splitlines() if x.strip()]
        if len(q)>=RAW_POOL_SIZE: return q[:RAW_POOL_SIZE]
    ds=load_dataset("allenai/c4","en",split="validation",streaming=True); q=[]
    for ex in tqdm(ds,desc="C4 B1"):
        text=ex.get("text","")
        if len(text)<2400: continue
        h=hashlib.sha256(text.encode()).hexdigest()
        if int(h[:8],16)%11!=1: continue
        q.append({"sha256":h,"text":text})
        if len(q)>=RAW_POOL_SIZE: break
    with open(p,"w") as f:
        for x in q: f.write(json.dumps(x)+"\n")
    return q
raw=get_b1(); frozen=load_repo_csv(spec["b1"]); dids=sorted(frozen.delta_id.unique())[:N_DELTAS]; design=frozen[frozen.delta_id.isin(dids)].copy()
for did,g in design.groupby("delta_id"):
    assert len(g)==R and g.receiver.nunique()==R and (g.receiver==g.origin_host).sum()==1
print("Frozen B1 deltas:",dids); design.to_csv(OUT/"FROZEN_B1_SELECTED_ROWS.csv",index=False)

HF_TOKEN=os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: HF_TOKEN=None

tok=AutoTokenizer.from_pretrained(spec["model_id"],token=HF_TOKEN,trust_remote_code=True,use_fast=True)
if tok.pad_token_id is None: tok.pad_token_id=tok.eos_token_id
ctx=[]
for x in raw:
    ids=tok(x["text"],add_special_tokens=False,truncation=False)["input_ids"]
    if len(ids)>=SEQ_LEN: ctx.append(torch.tensor(ids[:SEQ_LEN],dtype=torch.long))
    if len(ctx)>=N_CONTEXTS: break
assert len(ctx)==N_CONTEXTS

def layers(m): return m.model.layers
def mlp(m,i): return layers(m)[i].mlp
def indev(m):
    try: return next(m.model.embed_tokens.parameters()).device
    except Exception: return next(m.parameters()).device
@torch.inference_mode()
def capture(m,ids,li):
    a=[]; b=[]
    h1=mlp(m,li).register_forward_hook(lambda M,I,O:a.append(O[:,-1,:].detach().float().cpu()))
    def hh(M,I,O):
        z=O[0] if isinstance(O,(tuple,list)) else O; b.append(z.detach().float().cpu())
    h2=layers(m)[li].register_forward_hook(hh)
    try: m(input_ids=ids.to(indev(m)),use_cache=False,return_dict=True)
    finally: h1.remove(); h2.remove()
    return a[0],b[0]



In [ ]:
# ---------- Cache all 256 natural target-layer states in FP16 ----------
clear_cuda(); print("\nLoading full FP16 model for natural-state cache...")
m=AutoModelForCausalLM.from_pretrained(spec["model_id"],token=HF_TOKEN,trust_remote_code=True,torch_dtype=torch.float16,device_map="balanced",max_memory={0:"12000MiB",1:"12000MiB"},low_cpu_mem_usage=True,attn_implementation="eager")
m.eval()
for p in m.parameters(): p.requires_grad_(False)
Us=[]; Hs=[]
for s in tqdm(range(0,N_CONTEXTS,2),desc="cache 256 natural states"):
    ids=torch.stack(ctx[s:min(s+2,N_CONTEXTS)]); u,h=capture(m,ids,spec["layer"]); Us.append(u); Hs.append(h)
U=torch.cat(Us); H=torch.cat(Hs); print("U",U.shape,"H",H.shape)

# Protocol check against frozen natural displacement norms.
qa=[]
for did,g in design.groupby("delta_id"):
    i=int(g.origin_host.iloc[0]); j=int(g.origin_donor.iloc[0]); x=float((U[j]-U[i]).norm()); y=float(g.delta_norm.iloc[0]); qa.append([did,x,y,abs(x-y)/max(abs(y),1e-12)])
qa=pd.DataFrame(qa,columns=["delta_id","computed","frozen","relerr"]); qa.to_csv(OUT/"DELTA_NORM_PROTOCOL_CHECK.csv",index=False); print(qa)
assert qa.relerr.max()<.02, "Protocol/model/tokenizer drift >2%. STOP and send DELTA_NORM_PROTOCOL_CHECK.csv."
del m,Us,Hs; clear_cuda()



In [ ]:
# ---------- Descriptive PCA/5NN naturality proxy ----------
Un=U.numpy().astype(np.float64); pc=PCA(n_components=min(64,N_CONTEXTS-1,Un.shape[1]),random_state=SEED).fit(Un); Z=pc.transform(Un)
Dn=pairwise_distances(Z,Z); np.fill_diagonal(Dn,np.inf); nat5=np.sort(Dn,axis=1)[:,:5].mean(1); nmed=float(np.median(nat5)); np95=float(np.quantile(nat5,.95)); nr=[]
for did,g in design.groupby("delta_id"):
    i=int(g.origin_host.iloc[0]); j=int(g.origin_donor.iloc[0]); du=Un[j]-Un[i]
    for r in g.itertuples(index=False):
        k=int(r.receiver)
        for lam in DOSES:
            z=pc.transform((Un[k]+lam*du)[None])[0]; ds=np.linalg.norm(Z-z[None],axis=1); ds[k]=np.inf; d5=float(np.sort(ds)[:5].mean()); nr.append([did,k,k==i,lam,d5,d5/nmed,d5<=np95])
nat=pd.DataFrame(nr,columns=["delta_id","receiver","is_anchor","dose","d5","ratio","within_nat95"]); nat.to_csv(OUT/f"{MODEL_TO_RUN}_NATURALITY_ROWS.csv",index=False)
nats=nat[~nat.is_anchor].groupby("dose").agg(median_ratio=("ratio","median"),frac_within_nat95=("within_nat95","mean"),n=("ratio","size")).reset_index(); nats.to_csv(OUT/f"{MODEL_TO_RUN}_NATURALITY_SUMMARY.csv",index=False); print("\nNaturality proxy (descriptive only)\n",nats)



In [ ]:
# ---------- Build pruned FP32 downstream suffix across 2xT4 ----------
def make_suffix(model_id,li):
    print("\nLoading CPU FP16 model for pruning to downstream suffix...")
    x=AutoModelForCausalLM.from_pretrained(model_id,token=HF_TOKEN,trust_remote_code=True,torch_dtype=torch.float16,device_map={"":"cpu"},low_cpu_mem_usage=True,attn_implementation="eager")
    x.eval(); keep=list(x.model.layers[li+1:]); x.model.layers=torch.nn.ModuleList(keep)
    try: x.model.embed_tokens=None
    except Exception: pass
    gc.collect(); x=x.float(); n=len(x.model.layers); cut=n//2
    dm={f"model.layers.{i}":(0 if i<cut else 1) for i in range(n)}; dm["model.norm"]=1; dm["lm_head"]=1
    x=dispatch_model(x,device_map=dm); x.eval()
    for p in x.parameters(): p.requires_grad_(False)
    clear_cuda(); print("FP32 suffix layers",n,"split",cut,n-cut); return x
suf=make_suffix(spec["model_id"],spec["layer"])

def firstdev(x): return next(p for p in x.parameters() if p is not None).device
def slogit(x,h,du,a):
    d=firstdev(x); h=h.to(d,torch.float32); du=du.to(d,torch.float32); mask=torch.zeros_like(h); mask[:,-1,:]=du
    return x(inputs_embeds=h+a.to(d)*mask,use_cache=False,return_dict=True).logits[0,-1,:]
def exactjvp(x,h,du):
    d=firstdev(x); a=torch.zeros((),device=d,dtype=torch.float32,requires_grad=True)
    def f(v): return slogit(x,h,du,v)
    y,v=torch.autograd.functional.jvp(f,a,torch.ones_like(a),create_graph=False,strict=False); return y.detach().cpu(),v.detach().cpu()
@torch.inference_mode()
def dose_logits(x,h,du):
    d=firstdev(x); h=h.to(d,torch.float32); du=du.to(d,torch.float32); aa=torch.tensor(DOSES,device=d,dtype=torch.float32); B=len(DOSES)
    z=h.expand(B,-1,-1).clone(); z[:,-1,:]+=aa[:,None]*du[None]
    return x(inputs_embeds=z,use_cache=False,return_dict=True).logits[:,-1,:].detach().cpu()

def one_delta(did,g):
    i=int(g.origin_host.iloc[0]); j=int(g.origin_donor.iloc[0]); du=(U[j]-U[i]).float(); order=pd.concat([g[g.receiver==i],g[g.receiver!=i]],ignore_index=True)
    rows=[]; vh=ph=dzhost=None
    for rr in tqdm(list(order.itertuples(index=False)),desc=f"delta {did}",leave=False):
        k=int(rr.receiver); z0,v=exactjvp(suf,H[k:k+1],du); p=torch.softmax(z0.float(),-1); qfull=fisher(v,p); zz=dose_logits(suf,H[k:k+1],du); kd={lam:KL(z0,zz[t]) for t,lam in enumerate(DOSES)}
        if k==i: vh=v.clone(); ph=p.clone(); dzhost=(zz[list(DOSES).index(1.0)]-z0).clone()
        assert vh is not None and ph is not None and dzhost is not None
        rec={"delta_id":did,"origin_host":i,"origin_donor":j,"receiver":k,"is_anchor":k==i,"q_fp16_frozen":float(rr.q_fisher),"kl_fp16_frozen":float(rr.kl_cross),"q_exact_fp32":qfull,"q_p_only_fp32":fisher(vh,p),"q_J_only_fp32":fisher(v,ph),"kl_fixed_host_fullshift_fp32":KL(z0,z0+dzhost),"kl_fixed_host_jvp_fp32":KL(z0,z0+vh)}
        for lam,val in kd.items(): rec[f"kl_fp32_dose_{lam:.2f}"]=val
        rows.append(rec); del z0,v,p,zz; gc.collect()
    return pd.DataFrame(rows)

done=[]
for n,did in enumerate(dids,1):
    fp=OUT/"per_delta"/f"d{did:03d}.csv"
    if fp.exists() and len(pd.read_csv(fp))==R: x=pd.read_csv(fp); print("resume",did)
    else: x=one_delta(did,design[design.delta_id==did].copy()); x.to_csv(fp,index=False)
    done.append(x)
    if n%2==0: zip_checkpoint()
res=pd.concat(done,ignore_index=True); res.to_csv(OUT/f"{MODEL_TO_RUN}_EXACT_CROSSED_STRESS_ROWS.csv",index=False); del suf; clear_cuda(); print("completed",len(res),"rows")



In [ ]:
# ---------- Per-displacement primary/robustness summaries ----------
alt=res[~res.is_anchor].copy()
pairs={
 "exact_Q32_vs_frozen_paper_K":("q_exact_fp32","kl_fp16_frozen"),
 "frozen_Q16_vs_frozen_paper_K":("q_fp16_frozen","kl_fp16_frozen"),
 "frozen_Q16_vs_exact_Q32":("q_fp16_frozen","q_exact_fp32"),
 "exact_Q32_vs_FP32_K1":("q_exact_fp32","kl_fp32_dose_1.00"),
 "p_only_Q_vs_FP32_K1":("q_p_only_fp32","kl_fp32_dose_1.00"),
 "J_only_Q_vs_FP32_K1":("q_J_only_fp32","kl_fp32_dose_1.00"),
 "fixed_host_full_logit_vs_FP32_K1":("kl_fixed_host_fullshift_fp32","kl_fp32_dose_1.00"),
 "fixed_host_JVP_logit_vs_FP32_K1":("kl_fixed_host_jvp_fp32","kl_fp32_dose_1.00"),
 "frozen_K16_vs_FP32_K1":("kl_fp16_frozen","kl_fp32_dose_1.00")}
pr=[]
for did,g in alt.groupby("delta_id"):
    z={"delta_id":did}
    for nm,(a,b) in pairs.items(): z[nm]=rho(g[a],g[b])
    for lam in DOSES:
        c=f"kl_fp32_dose_{lam:.2f}"; z[f"Q32_vs_K_{lam:.2f}"]=rho(g.q_exact_fp32,g[c]); z[f"K_{lam:.2f}_vs_K1"]=rho(g[c],g["kl_fp32_dose_1.00"])
    pr.append(z)
pr=pd.DataFrame(pr); pr.to_csv(OUT/f"{MODEL_TO_RUN}_PER_DELTA_RHOS.csv",index=False)
sm=[]
for c in pr.columns[1:]:
    med,lo,hi=bootmed(pr[c],seed=SEED+sum(map(ord,c))); sm.append({"metric":c,"median_rho":med,"CI95_lo":lo,"CI95_hi":hi,"n_deltas":pr[c].notna().sum()})
sm=pd.DataFrame(sm); sm.to_csv(OUT/f"{MODEL_TO_RUN}_STRESS_SUMMARY.csv",index=False); print("\nStress-test summary\n",sm.to_string(index=False))



In [ ]:
# ---------- Figures ----------
dose=[]
for lam in DOSES:
    dose.append({"dose":lam,"Q32_vs_K":float(sm.loc[sm.metric==f"Q32_vs_K_{lam:.2f}","median_rho"].iloc[0]),"Kdose_vs_K1":float(sm.loc[sm.metric==f"K_{lam:.2f}_vs_K1","median_rho"].iloc[0])})
dp=pd.DataFrame(dose)
plt.figure(figsize=(6.4,4.2)); plt.plot(dp.dose,dp.Q32_vs_K,marker="o",label="Exact Q32 vs dose KL"); plt.plot(dp.dose,dp.Kdose_vs_K1,marker="o",label="Dose KL ordering vs full dose"); plt.xlabel("Dose lambda"); plt.ylabel("Median within-displacement Spearman rho"); plt.ylim(-1,1); plt.title(f"{MODEL_TO_RUN}: crossed dose robustness"); plt.legend(); plt.tight_layout(); plt.savefig(OUT/f"{MODEL_TO_RUN}_dose_robustness.png",dpi=220); plt.show()

names=["exact_Q32_vs_FP32_K1","p_only_Q_vs_FP32_K1","J_only_Q_vs_FP32_K1","fixed_host_full_logit_vs_FP32_K1"]
vals=[float(sm.loc[sm.metric==x,"median_rho"].iloc[0]) for x in names]
plt.figure(figsize=(7,4.2)); plt.bar(range(4),vals); plt.xticks(range(4),["Full Fisher","p-only","J-only","Fixed host\nlogit shift"]); plt.ylabel("Median within-displacement Spearman rho"); plt.ylim(-1,1); plt.title(f"{MODEL_TO_RUN}: receiver-ordering decomposition"); plt.tight_layout(); plt.savefig(OUT/f"{MODEL_TO_RUN}_geometry_decomposition.png",dpi=220); plt.show()



In [ ]:
# ---------- Compact factual summary with guardrails ----------
def gv(name):
    r=sm[sm.metric==name].iloc[0]; return r.median_rho,r.CI95_lo,r.CI95_hi
lines=[f"MODEL: {MODEL_TO_RUN}",f"B1 crossed audit: {N_DELTAS} fixed displacements x 15 alternative receivers",""]
for x in ["exact_Q32_vs_frozen_paper_K","frozen_Q16_vs_frozen_paper_K","frozen_Q16_vs_exact_Q32","exact_Q32_vs_FP32_K1","p_only_Q_vs_FP32_K1","J_only_Q_vs_FP32_K1","fixed_host_full_logit_vs_FP32_K1"]:
    a,b,c=gv(x); lines.append(f"{x}: median rho={a:.4f}, bootstrap95=[{b:.4f},{c:.4f}]")
lines += ["","DOSES:"]
for lam in DOSES:
    a,_,_=gv(f"Q32_vs_K_{lam:.2f}"); b,_,_=gv(f"K_{lam:.2f}_vs_K1"); lines.append(f"lambda={lam:.2f}: Q32-vs-K rho={a:.4f}; dose-order-vs-full rho={b:.4f}")
lines += ["","GUARDRAILS:",
"- Exact Q32 vs frozen paper KL is the direct crossed numerical audit of the paper's receiver-ranking claim.",
"- p-only/fixed-logit controls quantify receiver-native output-distribution geometry; J-only captures receiver-specific downstream response with host output geometry fixed.",
"- Small-dose persistence weakens but does not eliminate off-distribution concerns.",
"- PCA/5NN is descriptive only; it is not proof of being on-manifold.",
"- The crossed design varies whole natural receiver contexts; it does not isolate the final-token residual vector from all other context-dependent downstream information."]
txt="\n".join(lines); (OUT/"PAPER_READY_FACTUAL_SUMMARY.txt").write_text(txt); print("\n"+txt)

meta={"model":MODEL_TO_RUN,"repo_commit":REPO_COMMIT,"github_ref_used_preference":GITHUB_REF,"seed":SEED,"B1_bucket":1,"seq_len":64,"layer":spec["layer"],"n_deltas":N_DELTAS,"receiver_rows":R,"alt_receivers":R-1,"doses":list(DOSES),"suffix_precision":"FP32","prefix_states":"FP16","tf32":False}
(OUT/"RUN_METADATA.json").write_text(json.dumps(meta,indent=2))
(OUT/"README_RESULTS.txt").write_text("""Reviewer stress tests for the same workshop paper.\n\nKey outputs:\n- exact crossed FP32 downstream-suffix JVP scores on frozen B1 rows\n- p-only / J-only / fixed-logit geometry baselines\n- 0.10/0.25/0.50/1.00 dose KLs\n- held-out 72/24 generic receiver-fragility audit\n- anchor-row residual audit\n- descriptive PCA/5NN naturality proxy\n\nDo not interpret PCA/5NN as proof of naturality; do not call the suffix JVP full-model FP32; and do not claim the crossed design isolates the final-token vector from the whole receiver context.\n""")
zip_checkpoint(); final=Path("/kaggle/working")/(OUT.name+"_FINAL_RESULTS.zip")
if final.exists(): final.unlink()
with zipfile.ZipFile(final,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(OUT))
print("\nDONE. Download:",final)
print("Then rerun this same notebook in a fresh T4x2 session with MODEL_TO_RUN set to the other model.")
